# Chapter 12: Dates and Time Series

The programs from Chapter 12 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## Date Parts

`dates_basics.py`

In [1]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])
d = orders["order_date"]

print(d.min().date(), "to", d.max().date())
print(
    orders.assign(
        weekday=d.dt.day_name(),
        month=d.dt.month_name(),
        week=d.dt.isocalendar().week,
    )[["order_date", "weekday", "month", "week"]].head(3)
)

by_day = orders.groupby(d.dt.day_name())["order_id"].count()
order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
print(by_day.reindex(order))

2025-01-01 to 2025-12-31
  order_date    weekday    month  week
0 2025-01-01  Wednesday  January     1
1 2025-01-01  Wednesday  January     1
2 2025-01-01  Wednesday  January     1
order_date
Monday       815
Tuesday      815
Wednesday    799
Thursday     787
Friday       788
Saturday     960
Sunday       869
Name: order_id, dtype: int64


## Resampling: Days to Months

`resample.py`

In [2]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
daily = orders.set_index("order_date")["revenue"].resample("D").sum()

monthly = daily.resample("MS").sum()  # MS: month start labels
print((monthly / 1000).round(1).to_string())
print()
print("month-on-month change, last 3 months:")
print(monthly.pct_change().tail(3).round(3).to_string())

order_date
2025-01-01    22.6
2025-02-01    20.3
2025-03-01    26.0
2025-04-01    29.5
2025-05-01    29.0
2025-06-01    30.5
2025-07-01    29.3
2025-08-01    28.5
2025-09-01    30.8
2025-10-01    40.3
2025-11-01    49.0
2025-12-01    61.5
Freq: MS

month-on-month change, last 3 months:
order_date
2025-10-01    0.306
2025-11-01    0.215
2025-12-01    0.256
Freq: MS


## Rolling Averages

`rolling.py`

In [3]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", parse_dates=["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
daily = orders.set_index("order_date")["revenue"].resample("D").sum()

week = daily.rolling(7).mean()  # average of each day and the 6 before
print(
    pd.DataFrame({"daily": daily, "7-day average": week})
    .loc["2025-11-24":"2025-11-30"]
    .round(0)
)
print(
    "best 7 days ended on",
    week.idxmax().date(),
    f"with ${week.max():,.0f} a day",
)

             daily  7-day average
order_date                       
2025-11-24  1250.0         1659.0
2025-11-25  1476.0         1647.0
2025-11-26  2059.0         1766.0
2025-11-27  1753.0         1730.0
2025-11-28  1689.0         1689.0
2025-11-29  2317.0         1755.0
2025-11-30  2009.0         1793.0
best 7 days ended on 2025-12-12 with $2,359 a day
